# Stock Trader AI - Simple Training Pipeline

This notebook will train an LSTM model to predict stock prices. It's designed to be beginner-friendly and run on Google Colab.

## What this does:
1. Downloads stock data
2. Calculates technical indicators
3. Trains an LSTM model
4. Saves the trained model

Let's get started!


In [ ]:
# Step 1: Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install yfinance ta scikit-learn pandas numpy matplotlib seaborn

# Mount Google Drive to save your model
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Step 2: Import libraries and check if we have GPU
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Check if we have GPU (this makes training much faster!)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("Great! We have GPU acceleration!")
else:
    print("No GPU found, training will be slower but still work")


In [ ]:
# Step 3: Download stock data
# We'll use popular stocks for training
SYMBOLS = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'AMZN']

print("Downloading stock data...")
stock_data = {}

for symbol in SYMBOLS:
    print(f"Downloading {symbol}...")
    try:
        ticker = yf.Ticker(symbol)
        data = ticker.history(period="2y")  # 2 years of data
        if not data.empty:
            stock_data[symbol] = data
            print(f"✓ {symbol}: {len(data)} days of data")
        else:
            print(f"✗ {symbol}: No data")
    except Exception as e:
        print(f"✗ {symbol}: Error - {str(e)}")

print(f"\nSuccessfully downloaded data for {len(stock_data)} stocks")


In [ ]:
# Step 4: Calculate technical indicators
import ta

def add_technical_indicators(df):
    """Add technical indicators to stock data"""
    df = df.copy()
    
    # Moving averages
    df['SMA_20'] = ta.trend.sma_indicator(df['Close'], window=20)
    df['SMA_50'] = ta.trend.sma_indicator(df['Close'], window=50)
    
    # RSI (Relative Strength Index)
    df['RSI'] = ta.momentum.rsi(df['Close'], window=14)
    
    # MACD
    df['MACD'] = ta.trend.macd(df['Close'])
    df['MACD_signal'] = ta.trend.macd_signal(df['Close'])
    
    # Bollinger Bands
    bb = ta.volatility.BollingerBands(df['Close'])
    df['BB_upper'] = bb.bollinger_hband()
    df['BB_lower'] = bb.bollinger_lband()
    
    # Price ratios
    df['High_Low_ratio'] = df['High'] / df['Low']
    df['Close_Open_ratio'] = df['Close'] / df['Open']
    
    # Returns
    df['Returns'] = df['Close'].pct_change()
    df['Volatility'] = df['Returns'].rolling(window=20).std()
    
    return df

# Add indicators to all stocks
print("Calculating technical indicators...")
for symbol in stock_data:
    stock_data[symbol] = add_technical_indicators(stock_data[symbol])
    print(f"✓ Added indicators for {symbol}")

print("Technical indicators calculated!")


In [ ]:
# Step 5: Create a simple LSTM model
import torch.nn as nn

class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super(SimpleLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        
        self.fc = nn.Linear(hidden_size, 1)
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, x):
        # LSTM forward pass
        lstm_out, _ = self.lstm(x)
        
        # Take the last output
        last_output = lstm_out[:, -1, :]
        
        # Apply dropout and final layer
        output = self.dropout(last_output)
        output = self.fc(output)
        
        return output

print("LSTM model defined!")


In [ ]:
# Step 6: Prepare data for training
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

def prepare_data(stock_data, sequence_length=30):
    """Prepare data for LSTM training"""
    
    # Features we'll use
    feature_columns = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'SMA_20', 'SMA_50', 'RSI', 'MACD', 'MACD_signal',
        'BB_upper', 'BB_lower', 'High_Low_ratio', 'Close_Open_ratio',
        'Returns', 'Volatility'
    ]
    
    all_X, all_y = [], []
    
    for symbol, data in stock_data.items():
        print(f"Processing {symbol}...")
        
        # Select features and clean data
        features = data[feature_columns].fillna(method='ffill').fillna(0)
        
        # Target: next day's close price
        targets = data['Close'].shift(-1).values
        
        # Remove rows with NaN
        valid_indices = ~np.isnan(targets)
        features = features[valid_indices]
        targets = targets[valid_indices]
        
        # Normalize features
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        
        # Create sequences
        X, y = [], []
        for i in range(sequence_length, len(features_scaled)):
            X.append(features_scaled[i-sequence_length:i])
            y.append(targets[i])
        
        if len(X) > 0:
            all_X.extend(X)
            all_y.extend(y)
            print(f"  ✓ {symbol}: {len(X)} sequences")
    
    # Convert to numpy arrays
    X = np.array(all_X)
    y = np.array(all_y)
    
    # Normalize targets
    target_scaler = MinMaxScaler()
    y_scaled = target_scaler.fit_transform(y.reshape(-1, 1)).flatten()
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_scaled, test_size=0.2, random_state=42, shuffle=False
    )
    
    print(f"\nData prepared:")
    print(f"Total sequences: {len(X)}")
    print(f"Training: {len(X_train)}")
    print(f"Testing: {len(X_test)}")
    print(f"Features: {X.shape[2]}")
    
    return X_train, X_test, y_train, y_test, target_scaler

# Prepare the data
X_train, X_test, y_train, y_test, target_scaler = prepare_data(stock_data)
input_size = X_train.shape[2]


In [ ]:
# Step 7: Train the model
def train_model(X_train, y_train, X_test, y_test, epochs=50):
    """Train the LSTM model"""
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train)
    X_test_tensor = torch.FloatTensor(X_test)
    y_test_tensor = torch.FloatTensor(y_test)
    
    # Create data loaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    # Initialize model
    model = SimpleLSTM(input_size=input_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    # Training history
    train_losses = []
    test_losses = []
    
    print(f"Training model on {device}...")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs.squeeze(), batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                loss = criterion(outputs.squeeze(), batch_y)
                test_loss += loss.item()
        
        # Record losses
        train_losses.append(train_loss / len(train_loader))
        test_losses.append(test_loss / len(test_loader))
        
        # Print progress
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Train Loss: {train_losses[-1]:.6f}, Test Loss: {test_losses[-1]:.6f}")
    
    return model, train_losses, test_losses

# Train the model
model, train_losses, test_losses = train_model(X_train, y_train, X_test, y_test, epochs=30)


In [ ]:
# Step 8: Plot training results
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Validation Loss')
plt.title('Training Progress')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(test_losses, label='Validation Loss')
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Final training loss: {train_losses[-1]:.6f}")
print(f"Final validation loss: {test_losses[-1]:.6f}")


In [ ]:
# Step 9: Test the model and save it
def test_model(model, X_test, y_test, target_scaler):
    """Test the model and show results"""
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.FloatTensor(X_test).to(device)
        predictions = model(X_test_tensor).cpu().numpy().flatten()
    
    # Convert back to original scale
    predictions_actual = target_scaler.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test_actual = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    
    # Calculate metrics
    mse = mean_squared_error(y_test_actual, predictions_actual)
    mae = mean_absolute_error(y_test_actual, predictions_actual)
    r2 = r2_score(y_test_actual, predictions_actual)
    rmse = np.sqrt(mse)
    
    print(f"Model Performance:")
    print(f"RMSE: ${rmse:.2f}")
    print(f"MAE: ${mae:.2f}")
    print(f"R²: {r2:.4f}")
    
    # Plot predictions vs actual
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(y_test_actual[:100], label='Actual', alpha=0.8)
    plt.plot(predictions_actual[:100], label='Predicted', alpha=0.8)
    plt.title('Predictions vs Actual (First 100 samples)')
    plt.xlabel('Sample')
    plt.ylabel('Price ($)')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.scatter(y_test_actual, predictions_actual, alpha=0.6)
    plt.plot([y_test_actual.min(), y_test_actual.max()], 
             [y_test_actual.min(), y_test_actual.max()], 'r--', lw=2)
    plt.title('Actual vs Predicted')
    plt.xlabel('Actual Price ($)')
    plt.ylabel('Predicted Price ($)')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return predictions_actual, y_test_actual

# Test the model
predictions, actual = test_model(model, X_test, y_test, target_scaler)


In [ ]:
# Step 10: Save the trained model
import pickle

# Save model
model_path = '/content/drive/MyDrive/stock-trader/lstm_model.pth'
scaler_path = '/content/drive/MyDrive/stock-trader/target_scaler.pkl'

# Create directory if it doesn't exist
import os
os.makedirs('/content/drive/MyDrive/stock-trader', exist_ok=True)

# Save model state
torch.save({
    'model_state_dict': model.state_dict(),
    'input_size': input_size,
    'sequence_length': 30,
    'prediction_horizon': 1
}, model_path)

# Save scaler
with open(scaler_path, 'wb') as f:
    pickle.dump(target_scaler, f)

print(f"Model saved to: {model_path}")
print(f"Scaler saved to: {scaler_path}")

# Also save locally for download
torch.save({
    'model_state_dict': model.state_dict(),
    'input_size': input_size,
    'sequence_length': 30,
    'prediction_horizon': 1
}, '/content/lstm_model.pth')

with open('/content/target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

print("\nModel also saved locally for download:")
print("- lstm_model.pth")
print("- target_scaler.pkl")
print("\n🎉 Training completed successfully!")
print("\nNext steps:")
print("1. Download the model files")
print("2. Set up your database")
print("3. Deploy the API")
print("4. Test the system")
